# 벡터 스토어 기반 검색기 — LangChain v1

벡터 스토어를 `as_retriever()`로 감싸 동기·비동기·배치 호출이 가능한
Runnable 검색기로 사용합니다. 유사도, MMR, 점수 임계값, 동적 검색 설정을
다루고, 마지막에는 Upstage의 query/passage 자동 분리를 확인합니다.


> **2026-09-21 업데이트**
>
> 이 노트북은 `langchain 1.4.2`, `langchain-core 1.6.3`,
> `langchain-openai 1.6.2`, `langchain-chroma 1.1.0` 기준으로 다시 작성했습니다.
> LangChain v1에서 예전 `langchain.retrievers` 구현은 `langchain-classic`으로
> 이동했고 `langchain-community`도 보관 상태이므로, 새 코드에서는 두 패키지와
> `langchain-teddynote`에 의존하지 않습니다. 대신 `langchain-core`의 Runnable,
> 공급자별 파트너 패키지, 명시적인 검색 함수를 조합합니다.
>
> 공식 참고: [LangChain v1 변경 사항](https://docs.langchain.com/oss/python/releases/langchain-v1),
> [v1 마이그레이션](https://docs.langchain.com/oss/python/migrate/langchain-v1),
> [OpenAI 임베딩](https://docs.langchain.com/oss/python/integrations/embeddings/openai),
> [Chroma 통합](https://docs.langchain.com/oss/python/integrations/vectorstores/chroma)


In [ ]:
# 최초 1회만 주석을 해제하여 설치하세요.
# %pip install -qU "langchain==1.4.2" "langchain-core==1.6.3" \
#   "langchain-openai==1.6.2" "langchain-chroma==1.1.0" \
#   "langchain-text-splitters==1.1.2" python-dotenv


In [ ]:
import getpass
import os
from pathlib import Path
from typing import Any, Literal, TypedDict
from uuid import uuid4

from dotenv import load_dotenv
from langchain_chroma import Chroma
from langchain_core.documents import Document
from langchain_core.runnables import RunnableLambda
from langchain_openai import OpenAIEmbeddings
from langchain_text_splitters import CharacterTextSplitter

load_dotenv()
if not os.getenv("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("OPENAI_API_KEY: ")

os.environ.setdefault("LANGSMITH_PROJECT", "CH10-Retriever-Modern")
# 추적을 사용하려면 .env에 LANGSMITH_TRACING=true와 LANGSMITH_API_KEY를 설정하세요.

CHROMA_CONFIGURATION = {"hnsw": {"space": "cosine"}}


def bounded_cosine_relevance(distance: float) -> float:
    # Chroma cosine distance(0~2)를 직관적인 relevance(1~0)로 변환합니다.
    return max(0.0, min(1.0, 1.0 - distance / 2.0))


## 문서 로드·분할·인덱싱

단순 텍스트 파일은 보관 상태인 범용 loader 패키지 대신 표준 `pathlib`로
읽고 `Document`로 감쌉니다. 인코딩과 메타데이터가 코드에 명확히 드러납니다.


In [ ]:
data_path = Path("data/appendix-keywords.txt")
if not data_path.exists():
    raise FileNotFoundError(f"실습 파일이 없습니다: {data_path.resolve()}")

source_docs = [
    Document(
        page_content=data_path.read_text(encoding="utf-8"),
        metadata={"source": str(data_path)},
    )
]
splitter = CharacterTextSplitter(chunk_size=300, chunk_overlap=0)
chunks = splitter.split_documents(source_docs)

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    collection_name=f"vectorstore-demo-{uuid4().hex}",
    collection_configuration=CHROMA_CONFIGURATION,
    relevance_score_fn=bounded_cosine_relevance,
)
print(f"인덱싱한 청크 수: {len(chunks)}")


## 기본 유사도 검색

Retriever는 Runnable이므로 `invoke`, `ainvoke`, `batch`, `abatch`를 같은
인터페이스로 사용할 수 있습니다.


In [ ]:
similarity_retriever = vectorstore.as_retriever(search_kwargs={"k": 4})
docs = similarity_retriever.invoke("임베딩(Embedding)은 무엇인가요?")

for index, doc in enumerate(docs, start=1):
    print(f"[{index}] {doc.page_content}\n{'-' * 80}")


## MMR(Maximal Marginal Relevance)

`fetch_k`개 후보에서 관련성과 결과 간 다양성을 함께 고려해 `k`개를 고릅니다.
`lambda_mult=1`에 가까울수록 질의 유사성을, `0`에 가까울수록 다양성을 더
중시합니다. 오래된 예제의 반대 설명을 바로잡았습니다.


In [ ]:
mmr_retriever = vectorstore.as_retriever(
    search_type="mmr",
    search_kwargs={"k": 2, "fetch_k": 10, "lambda_mult": 0.6},
)
mmr_retriever.invoke("임베딩(Embedding)은 무엇인가요?")


## 유사도 점수 임계값과 top-k

임계값은 벡터 스토어의 relevance-score 변환 방식에 의존합니다. 운영 환경에서는
고정값을 복사하지 말고 검증 데이터로 보정하세요.


In [ ]:
threshold_retriever = vectorstore.as_retriever(
    search_type="similarity_score_threshold",
    search_kwargs={"k": 4, "score_threshold": 0.5},
)
threshold_docs = threshold_retriever.invoke("Word2Vec은 무엇인가요?")
print(f"임계값을 통과한 문서 수: {len(threshold_docs)}")

top1_retriever = vectorstore.as_retriever(search_kwargs={"k": 1})
print(top1_retriever.invoke("임베딩은 무엇인가요?")[0].page_content)


## 동적 검색 설정: 입력 스키마가 드러나는 Runnable

예전처럼 내부 Pydantic 필드를 `configurable_fields()`로 노출하기보다,
검색 정책을 명시적인 입력으로 받으면 API 스키마와 검증 지점을 이해하기 쉽습니다.


In [ ]:
class SearchRequest(TypedDict, total=False):
    query: str
    search_type: Literal["similarity", "mmr", "similarity_score_threshold"]
    search_kwargs: dict[str, Any]


def retrieve(request: SearchRequest) -> list[Document]:
    search_type = request.get("search_type", "similarity")
    search_kwargs = request.get("search_kwargs", {"k": 4})
    return vectorstore.as_retriever(
        search_type=search_type,
        search_kwargs=search_kwargs,
    ).invoke(request["query"])


configurable_search = RunnableLambda(retrieve).with_config(
    {"run_name": "configurable_vector_search"}
)

requests = [
    {"query": "Word2Vec은 무엇인가요?", "search_kwargs": {"k": 3}},
    {
        "query": "Word2Vec은 무엇인가요?",
        "search_type": "mmr",
        "search_kwargs": {"k": 2, "fetch_k": 10, "lambda_mult": 0.3},
    },
]
results = configurable_search.batch(requests)
[len(result) for result in results]


## Upstage query/passage 임베딩

최신 `langchain-upstage`에서는 모델명에 `-query`, `-passage`를 붙이지 않습니다.
`embed_query()`와 `embed_documents()` 호출에 따라 통합 클래스가 올바른 접미사를
자동 적용하므로 동일한 객체를 벡터 스토어에 전달합니다.


In [ ]:
# 최초 1회만 주석을 해제하세요.
# %pip install -qU "langchain-upstage==0.7.7"

import getpass
from langchain_upstage import UpstageEmbeddings

if not os.getenv("UPSTAGE_API_KEY"):
    os.environ["UPSTAGE_API_KEY"] = getpass.getpass("UPSTAGE_API_KEY: ")

upstage_embeddings = UpstageEmbeddings(model="solar-embedding-1-large")
upstage_store = Chroma.from_documents(
    documents=chunks,
    embedding=upstage_embeddings,
    collection_name=f"upstage-demo-{uuid4().hex}",
    collection_configuration=CHROMA_CONFIGURATION,
    relevance_score_fn=bounded_cosine_relevance,
)
upstage_store.similarity_search("임베딩은 무엇인가요?", k=2)
